
# GLOBAL HEALTH INTELLIGENCE
### A Data-Driven Analysis of Global Healthcare Indicators

**Objective:** To investigate relationships between healthcare expenditure, life expectancy and infant mortality across countries.

**Data source:** World Bank Open Data

**Tools:** Python, Pandas, Plotly


In [3]:

import pandas as pd
import requests

print("Connecting to World Bank Health Database...")

indicators = {
    "SP.DYN.LE00.IN": "Life_Expectancy",
    "SP.DYN.IMRT.IN": "Infant_Mortality",
    "SH.XPD.CHEX.GD.ZS": "Health_Expenditure_GDP"
}

all_data = []

for code, name in indicators.items():

    url = (
        f"https://api.worldbank.org/v2/country/all/"
        f"indicator/{code}?format=json&date=2015:2023&per_page=20000"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    records = response.json()[1]

    df = pd.DataFrame([
        {
            "Country": item["country"]["value"],
            "Country_Code": item["countryiso3code"],
            "Year": int(item["date"]),
            name: item["value"]
        }
        for item in records
        if item["value"] is not None
        and item["countryiso3code"]
    ])

    all_data.append(df)

    print(name, "downloaded:", len(df), "records")

print("Data acquisition complete!")


Connecting to World Bank Health Database...
Life_Expectancy downloaded: 2349 records
Infant_Mortality downloaded: 2160 records
Health_Expenditure_GDP downloaded: 2126 records
Data acquisition complete!


In [4]:

# STEP 3: DATA CLEANING AND MERGING

# Merge the three datasets
health_data = all_data[0].merge(
    all_data[1],
    on=["Country", "Country_Code", "Year"],
    how="outer"
)

health_data = health_data.merge(
    all_data[2],
    on=["Country", "Country_Code", "Year"],
    how="outer"
)

# Remove regional aggregates using ISO3 country codes
health_data = health_data[
    health_data["Country_Code"].str.len() == 3
].copy()

# Sort records
health_data = health_data.sort_values(
    ["Country", "Year"]
).reset_index(drop=True)

# Inspect the dataset
print("Dataset shape:", health_data.shape)

print("\nFirst 10 records:")
display(health_data.head(10))

print("\nMissing values:")
print(health_data.isnull().sum())


Dataset shape: (2349, 6)

First 10 records:


,Country,Country_Code,Year,Life_Expectancy,Infant_Mortality,Health_Expenditure_GDP
0,Afghanistan,AFG,2015,62.270000,64.000000,10.105348
1,Afghanistan,AFG,2016,62.646000,61.900000,11.818590
2,Afghanistan,AFG,2017,62.406000,60.100000,12.620817
3,Afghanistan,AFG,2018,62.443000,58.300000,14.208419
4,Afghanistan,AFG,2019,62.941000,56.600000,14.831320
5,Afghanistan,AFG,2020,61.454000,54.900000,15.533614
6,Afghanistan,AFG,2021,60.417000,53.200000,21.508444
7,Afghanistan,AFG,2022,65.617000,51.600000,23.088169
8,Afghanistan,AFG,2023,66.035000,49.900000,14.985763
9,Africa Eastern and Southern,AFE,2015,61.713031,48.237049,5.891588



Missing values:
Country                     0
Country_Code                0
Year                        0
Life_Expectancy             0
Infant_Mortality          189
Health_Expenditure_GDP    223
dtype: int64


In [5]:

# BASIC DATASET PROFILE

print("Number of countries:",
      health_data["Country_Code"].nunique())

print("Years covered:",
      sorted(health_data["Year"].unique()))

print("\nStatistical summary:")
display(health_data.describe().round(2))


Number of countries: 261
Years covered: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

Statistical summary:


,Year,Life_Expectancy,Infant_Mortality,Health_Expenditure_GDP
count,2349.00,2349.00,2160.00,2126.00
mean,2019.00,72.34,22.29,6.63
std,2.58,7.64,19.95,3.02
min,2015.00,18.82,1.30,1.33
25%,2017.00,66.94,6.34,4.46
50%,2019.00,73.00,15.20,5.95
75%,2021.00,77.81,34.61,8.32
max,2023.00,86.37,206.80,27.09


In [6]:

# STEP 4: FILTER ACTUAL COUNTRIES

country_url = (
    "https://api.worldbank.org/v2/country"
    "?format=json&per_page=400"
)

country_response = requests.get(country_url, timeout=30)
country_response.raise_for_status()

country_records = country_response.json()[1]

# Keep countries with an actual World Bank geographic region
valid_country_codes = {
    item["id"]
    for item in country_records
    if item["region"]["id"] != "NA"
    and item["iso2Code"]
}

clean_data = health_data[
    health_data["Country_Code"].isin(valid_country_codes)
].copy()

print("Actual countries/territories:",
      clean_data["Country_Code"].nunique())

print("Clean dataset shape:", clean_data.shape)

# Investigate unusually low life expectancy
print("\nLife expectancy below 40 years:")
display(
    clean_data[
        clean_data["Life_Expectancy"] < 40
    ][["Country", "Year", "Life_Expectancy"]]
)

# Check missing values
print("\nMissing values after filtering:")
print(clean_data.isnull().sum())


Actual countries/territories: 217
Clean dataset shape: (1953, 6)

Life expectancy below 40 years:


,Country,Year,Life_Expectancy
373,Central African Republic,2019,31.530
376,Central African Republic,2022,18.818
1962,South Sudan,2015,39.757
1963,South Sudan,2016,36.720
1964,South Sudan,2017,35.351



Missing values after filtering:
Country                     0
Country_Code                0
Year                        0
Life_Expectancy             0
Infant_Mortality          189
Health_Expenditure_GDP    223
dtype: int64


In [7]:

import plotly.express as px

# Average life expectancy by year
yearly_life = (
    clean_data.groupby("Year")["Life_Expectancy"]
    .mean()
    .reset_index()
)

fig = px.line(
    yearly_life,
    x="Year",
    y="Life_Expectancy",
    markers=True,
    title="Global Life Expectancy Trends (2015–2023)",
    labels={
        "Year": "Year",
        "Life_Expectancy": "Life Expectancy (Years)"
    }
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()


In [8]:

# STEP 7: HANDLE ANOMALOUS VALUES

# Identify suspicious observations
anomalies = clean_data[
    clean_data["Life_Expectancy"] < 40
]

print("Observations flagged for investigation:")
display(anomalies[
    ["Country", "Year", "Life_Expectancy"]
])

# Exclude flagged observations from this analysis only
analysis_data = clean_data[
    clean_data["Life_Expectancy"] >= 40
].copy()

print("Original records:", len(clean_data))
print("Analysis records:", len(analysis_data))

# Calculate missing data percentages
missing_percentage = (
    analysis_data.isnull().mean() * 100
).round(2)

print("\nMissing data percentage:")
display(missing_percentage)


Observations flagged for investigation:


,Country,Year,Life_Expectancy
373,Central African Republic,2019,31.530
376,Central African Republic,2022,18.818
1962,South Sudan,2015,39.757
1963,South Sudan,2016,36.720
1964,South Sudan,2017,35.351


Original records: 1953
Analysis records: 1948

Missing data percentage:


,0
Country,0.00
Country_Code,0.00
Year,0.00
Life_Expectancy,0.00
Infant_Mortality,9.70
Health_Expenditure_GDP,11.34


In [9]:

import plotly.express as px

yearly_life = (
    analysis_data.groupby("Year")["Life_Expectancy"]
    .mean()
    .reset_index()
)

fig = px.line(
    yearly_life,
    x="Year",
    y="Life_Expectancy",
    markers=True,
    title="Average Country Life Expectancy (2015–2023)"
)

fig.update_layout(template="plotly_white")
fig.show()


In [10]:

complete_data = analysis_data.dropna(
    subset=["Health_Expenditure_GDP", "Life_Expectancy"]
)

fig = px.scatter(
    complete_data,
    x="Health_Expenditure_GDP",
    y="Life_Expectancy",
    color="Year",
    hover_name="Country",
    title="Healthcare Expenditure vs Life Expectancy",
    labels={
        "Health_Expenditure_GDP": "Health Expenditure (% GDP)",
        "Life_Expectancy": "Life Expectancy (Years)"
    }
)

fig.update_layout(template="plotly_white")
fig.show()


In [11]:

correlation_data = analysis_data[
    ["Life_Expectancy", "Infant_Mortality",
     "Health_Expenditure_GDP"]
]

correlation_matrix = correlation_data.corr()

print("Correlation Matrix:")
display(correlation_matrix.round(3))

fig = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Correlation Between Global Health Indicators",
    aspect="auto"
)

fig.show()


Correlation Matrix:


,Life_Expectancy,Infant_Mortality,Health_Expenditure_GDP
Life_Expectancy,1.000,-0.904,0.249
Infant_Mortality,-0.904,1.000,-0.270
Health_Expenditure_GDP,0.249,-0.270,1.000


In [12]:

# STEP 9: CORRELATION WITH SAMPLE SIZES

pairs = [
    ("Life_Expectancy", "Infant_Mortality"),
    ("Life_Expectancy", "Health_Expenditure_GDP"),
    ("Infant_Mortality", "Health_Expenditure_GDP")
]

for x, y in pairs:

    pair_data = analysis_data[[x, y]].dropna()

    pearson = pair_data[x].corr(pair_data[y])
    spearman = pair_data[x].corr(pair_data[y], method="spearman")

    print(f"\n{x} vs {y}")
    print("Observations:", len(pair_data))
    print("Pearson correlation:", round(pearson, 3))
    print("Spearman correlation:", round(spearman, 3))



Life_Expectancy vs Infant_Mortality
Observations: 1759
Pearson correlation: -0.904
Spearman correlation: -0.921

Life_Expectancy vs Health_Expenditure_GDP
Observations: 1727
Pearson correlation: 0.249
Spearman correlation: 0.36

Infant_Mortality vs Health_Expenditure_GDP
Observations: 1718
Pearson correlation: -0.27
Spearman correlation: -0.426


In [13]:

# Compare countries with data in both 2019 and 2021

comparison = analysis_data[
    analysis_data["Year"].isin([2019, 2021])
].pivot_table(
    index="Country",
    columns="Year",
    values="Life_Expectancy"
).dropna()

comparison["Change"] = (
    comparison[2021] - comparison[2019]
)

print("Countries compared:", len(comparison))

print("Average change:",
      round(comparison["Change"].mean(), 3), "years")

print("\nLargest decreases:")
display(comparison.nsmallest(5, "Change"))

print("\nLargest increases:")
display(comparison.nlargest(5, "Change"))


Countries compared: 216
Average change: -1.277 years

Largest decreases:


Year,2019,2021,Change
Country,,,
Bolivia,67.820,61.427,-6.393
Paraguay,73.668,68.108,-5.560
Mexico,74.530,69.750,-4.780
Guyana,69.067,64.323,-4.744
Peru,76.275,71.596,-4.679



Largest increases:


Year,2019,2021,Change
Country,,,
Syrian Arab Republic,70.970000,72.418000,1.448000
"Congo, Rep.",63.152000,64.192000,1.040000
Samoa,70.269000,71.096000,0.827000
Sierra Leone,59.593000,60.260000,0.667000
Mongolia,71.053171,71.645122,0.591951


In [14]:

import plotly.express as px

# Select five largest decreases and five largest increases
largest_changes = pd.concat([
    comparison.nsmallest(5, "Change"),
    comparison.nlargest(5, "Change")
]).reset_index()

largest_changes["Impact"] = largest_changes["Change"].apply(
    lambda x: "Decrease" if x < 0 else "Increase"
)

fig = px.bar(
    largest_changes.sort_values("Change"),
    x="Change",
    y="Country",
    color="Impact",
    orientation="h",
    text="Change",
    title="Largest Changes in Life Expectancy (2019–2021)",
    labels={
        "Change": "Change in Life Expectancy (Years)",
        "Country": "Country"
    },
    color_discrete_map={
        "Decrease": "#D95F59",
        "Increase": "#269B79"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=True
)

fig.show()


In [15]:

# Export analysis dataset
analysis_data.to_csv(
    "global_health_analysis.csv",
    index=False
)

# Export country comparison
comparison.to_csv(
    "life_expectancy_2019_2021_comparison.csv"
)

# Save interactive charts
fig.write_html(
    "life_expectancy_country_comparison.html"
)

print("Project files created successfully!")


Project files created successfully!


In [16]:

from google.colab import files

files.download("global_health_analysis.csv")
files.download("life_expectancy_2019_2021_comparison.csv")
files.download("life_expectancy_country_comparison.html")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:

import plotly.express as px

largest_changes = pd.concat([
    comparison.nsmallest(5, "Change"),
    comparison.nlargest(5, "Change")
]).reset_index()

largest_changes["Impact"] = largest_changes["Change"].apply(
    lambda x: "Decrease" if x < 0 else "Increase"
)

largest_changes["Label"] = largest_changes["Change"].map(
    lambda x: f"{x:+.2f}"
)

largest_changes = largest_changes.sort_values("Change")

fig = px.bar(
    largest_changes,
    x="Change",
    y="Country",
    color="Impact",
    orientation="h",
    text="Label",
    title="Changes in Life Expectancy Across Countries (2019–2021)",
    color_discrete_map={
        "Decrease": "#D95F59",
        "Increase": "#269B79"
    }
)

fig.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Change in Life Expectancy (Years)",
    yaxis_title="Country",
    xaxis=dict(range=[-7.5, 2.2]),
    margin=dict(l=130, r=100, t=90, b=70),
    height=600,
    font=dict(size=13)
)

fig.show()

fig.write_html("life_expectancy_country_comparison.html")
